In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Simulación exacta con las primitivas del SDK de Qiskit

<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página fue desarrollado usando los siguientes requisitos.
Recomendamos usar estas versiones o más recientes.

```
qiskit[all]~=2.3.0
```
</details>
Las primitivas de referencia del SDK de Qiskit realizan simulaciones locales de vector de estado. Estas simulaciones no admiten
modelar el ruido del dispositivo, pero son útiles para prototipar algoritmos rápidamente antes de explorar técnicas de simulación
más avanzadas ([usando Qiskit Aer](./simulate-stabilizer-circuits)) o ejecutar en dispositivos reales ([primitivas de Qiskit Runtime](primitives)).

La primitiva Estimator puede calcular valores de expectación de circuitos, y la primitiva Sampler puede muestrear
desde las distribuciones de salida de los circuitos.

Las siguientes secciones muestran cómo usar las primitivas de referencia para ejecutar tu flujo de trabajo localmente.

## Usa el Estimator de referencia
La implementación de referencia de `EstimatorV2` en `qiskit.primitives` que se ejecuta sobre simuladores locales de vector de estado
es la clase [`StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator). Puede tomar circuitos, observables y parámetros como entradas y devuelve los valores de expectación calculados localmente.

El siguiente código prepara las entradas que se usarán en los ejemplos a continuación. El tipo de entrada esperado para los
observables es [`qiskit.quantum_info.SparsePauliOp`](../api/qiskit/qiskit.quantum_info.SparsePauliOp). Ten en cuenta que
el circuito del ejemplo está parametrizado, pero también puedes ejecutar el Estimator en circuitos no parametrizados.

> **Note:** Cualquier circuito que se pase a un Estimator **no** debe incluir ninguna **medición**.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# circuit for which you want to obtain the expected value
circuit = QuantumCircuit(2)
circuit.ry(Parameter("theta"), 0)
circuit.h(0)
circuit.cx(0, 1)
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/5b41a52d-8f15-4ce4-b3f6-effd91946d9c-0.svg" alt="Output of the previous code cell" />

In [2]:
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# observable(s) whose expected values you want to compute

observable = SparsePauliOp(["II", "XX", "YY", "ZZ"], coeffs=[1, 1, -1, 1])

# value(s) for the circuit parameter(s)
parameter_values = [[0], [np.pi / 6], [np.pi / 2]]

> **Tip:** El flujo de trabajo de las primitivas de Qiskit Runtime requiere que los circuitos y observables sean transformados para usar únicamente instrucciones compatibles con la QPU (denominados circuitos y observables de *arquitectura de conjunto de instrucciones (ISA)*). Las primitivas de referencia aún aceptan instrucciones abstractas, ya que dependen de simulaciones locales de vector de estado, pero transpilar el circuito puede resultar beneficioso en términos de optimización del circuito.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(circuit)
>   isa_observable = observable.apply_layout(isa_circuit.layout)
>   ```

### Inicializa el Estimator
Instancia un [`qiskit.primitives.StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator).

In [3]:
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

### Ejecuta y obtén resultados
Este ejemplo solo usa un circuito (de tipo [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit)) y un
observable.

Ejecuta la estimación llamando al método [`StatevectorEstimator.run`](../api/qiskit/qiskit.primitives.StatevectorEstimator#run), que devuelve una instancia de un objeto [`PrimitiveJob`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveJob). Puedes obtener los resultados del trabajo (como un objeto [`qiskit.primitives.PrimitiveResult`](../api/qiskit/qiskit.primitives.PrimitiveResult))
con el método [`qiskit.primitives.PrimitiveJob.result`](../api/qiskit/qiskit.primitives.PrimitiveJob#result).

In [4]:
job = estimator.run([(circuit, observable, parameter_values)])
result = job.result()
print(f" > Result class: {type(result)}")

 > Result class: <class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>


#### Get the expected value from the result

The primitives result outputs an array of [`PubResult`](/docs/api/qiskit/qiskit.primitives.PubResult#pubresult) objects, where each item of the array is a `PubResult` object that contains in its data the array of evaluations corresponding to every circuit-observable combination in the PUB.

To retrieve the expectation values and metadata for the first (and in this case, only) circuit evaluation, we must access the evaluation [`data`](/docs/api/qiskit/qiskit.primitives.PubResult#data) for PUB 0:

In [5]:
print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: [4.         3.73205081 2.        ]
 > Metadata: {'target_precision': 0.0, 'circuit_metadata': {}}


#### Obtén el valor esperado del resultado
La salida del resultado de las primitivas es un array de objetos [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult), donde cada elemento del array es un objeto `PubResult` que contiene en sus datos el array de evaluaciones correspondiente a cada combinación circuito-observable del PUB.

Para recuperar los valores de expectación y los metadatos de la primera (y en este caso única) evaluación del circuito, debemos acceder a los [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#data) de evaluación para el PUB 0:

In [6]:
# Estimate expectation values for two PUBs, both with 0.05 precision.
precise_job = estimator.run(
    [(circuit, observable, parameter_values)], precision=0.05
)

For a full example, see the [Primitives examples](primitives-examples#estimator-examples) page.

## Use the reference Sampler

The reference implementations of `SamplerV2` in `qiskit.primitives` is the [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) class. It takes circuits and parameters as inputs and returns the results from sampling from the output probability distributions as a quasi-probability distribution of output states.

The following code prepares the inputs used in the examples that follow. Note that
these examples run a single parametrized circuit, but you can also run the Sampler
on non-parametrized circuits.

In [7]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg" alt="Output of the previous code cell" />

### Configura las opciones de ejecución del Estimator
Por defecto, el Estimator de referencia realiza un cálculo exacto del vector de estado basado en la clase
[`quantum_info.Statevector`](../api/qiskit/qiskit.quantum_info.Statevector).
Sin embargo, esto puede modificarse para introducir el efecto del overhead de muestreo (también conocido como "ruido de shot").

El Estimator acepta un argumento `precision` que expresa las barras de error que la
implementación primitiva debe alcanzar para las estimaciones de valores de expectación. Este es el overhead de muestreo y se define exclusivamente en el método `.run()`. Esto te permite ajustar la opción hasta el nivel de PUB.

In [8]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

Para ver un ejemplo completo, consulta la página de [ejemplos de Primitivas](primitives-examples#estimator-examples).

## Usa el Sampler de referencia
La implementación de referencia de `SamplerV2` en `qiskit.primitives` es la clase [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler). Toma circuitos y parámetros como entradas y devuelve los resultados del muestreo de las distribuciones de probabilidad de salida como una distribución de cuasi-probabilidad de los estados de salida.

El siguiente código prepara las entradas utilizadas en los ejemplos a continuación. Ten en cuenta que
estos ejemplos ejecutan un único circuito parametrizado, pero también puedes ejecutar el Sampler
en circuitos no parametrizados.

In [9]:
# execute 1 circuit with Sampler
job = sampler.run([circuit])
pub_result = job.result()[0]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg)

> **Note:** Cualquier circuito cuántico que se pase a un Sampler **debe** incluir mediciones.

> **Tip:** El flujo de trabajo de las primitivas de Qiskit Runtime requiere que los circuitos sean transformados para usar únicamente instrucciones compatibles con la QPU (denominados circuitos ISA). Las primitivas de referencia aún aceptan instrucciones abstractas, ya que dependen de simulaciones locales de vector de estado, pero transpilar el circuito puede resultar beneficioso en términos de optimización del circuito.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(qc)
>   ```

### Inicializa `SamplerV2`
Instancia [`qiskit.primitives.StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler):

In [10]:
from qiskit.transpiler import generate_preset_pass_manager

# create two circuits
circuit1 = circuit.copy()
circuit2 = circuit.copy()

# transpile circuits
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit1 = pm.run(circuit1)
isa_circuit2 = pm.run(circuit2)
# execute 2 circuits using Sampler
job = sampler.run([(isa_circuit1), (isa_circuit2)])
pub_result_1 = job.result()[0]
pub_result_2 = job.result()[1]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


### Ejecuta y obtén resultados

In [11]:
# Define quantum circuit with 2 qubits
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw()

        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 

In [12]:
# Transpile circuit
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit = pm.run(circuit)
# Run using sampler
result = sampler.run([circuit]).result()
# Access result data for PUB 0
data_pub = result[0].data
# Access bitstring for the classical register "meas"
bitstrings = data_pub.meas.get_bitstrings()
print(f"The number of bitstrings is: {len(bitstrings)}")
# Get counts for the classical register "meas"
counts = data_pub.meas.get_counts()
print(f"The counts are: {counts}")

The number of bitstrings is: 1024
The counts are: {'11': 515, '00': 509}


Las primitivas aceptan múltiples PUBs como entradas, y cada PUB obtiene su propio resultado. Por lo tanto, puedes ejecutar diferentes circuitos con distintas combinaciones de parámetros/observables y recuperar los resultados de los PUBs:

In [13]:
# Sample two circuits at 128 shots each.
sampler.run([isa_circuit1, isa_circuit2], shots=128)
# Sample two circuits at different amounts of shots. The "None"s are necessary
# as placeholders
# for the lack of parameter values in this example.
sampler.run([(isa_circuit1, None, 123), (isa_circuit2, None, 456)])

For a full example, see the [Primitives examples](./primitives-examples#sampler-examples) page.
## Next steps

<Admonition type="tip" title="Recommendations">
  - For higher-performance simulation that can handle larger circuits, or to incorporate noise models into your simulation, see [Exact and noisy simulation with Qiskit Aer primitives](simulate-with-qiskit-aer).
  - To learn how to use Quantum Composer for simulation, see the [IBM Quantum Composer](/docs/guides/composer) guide.
  - Read the [Qiskit Estimator API](/docs/api/qiskit/1.4/qiskit.primitives.Estimator) reference.
  - Read the [Qiskit Sampler API](/docs/api/qiskit/1.4/qiskit.primitives.Sampler) reference.
  - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>